## Notebook10c

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds-py/refs/heads/main/funs.py

In [ ]:
import numpy as np
import polars as pl

from funs import *
from plotnine import *
from polars import col as c
theme_set(theme_minimal())
pl.Config(tbl_rows=20)

ub = "https://raw.githubusercontent.com/taylor-arnold/fds-py-nb/refs/heads/main/"

In [ ]:
g1_orig = pl.read_csv(ub + "data/game01.csv")
g1_meta = pl.read_csv(ub + "data/game01-meta.csv")
g2_orig = pl.read_csv(ub + "data/game02.csv")
g2_meta = pl.read_csv(ub + "data/game02-meta.csv")
g3 = pl.read_csv(ub + "data/game03.csv")
g3_meta = pl.read_csv(ub + "data/game03-meta.csv")

### Game 1: 90 percent ranges

1. We start by doing some data cleaning. In this case, we need to convert the metric answers into inches so that all of our data is on the same scale. We save the dataset as `g1` and then are ready to do some analysis.

In [ ]:
g1 = (
    g1_orig
    .with_columns(
        low = pl.when(c.unit == "cm").then(c.low * 0.3937).otherwise(c.low),
        high = pl.when(c.unit == "cm").then(c.high * 0.3937).otherwise(c.high)
    )
)
g1

2. Join `g1` to the `g1_meta` data using the key `c.question`. Create a new column called `correct` that is True when the answer is in the bounds and False otherwise. Group by student and compute the average value of the correct variable (this is the proportion of correct responses). Sort the output by score.

3. Repeat what you did above, but this time group by `question_desc` to see how difficult each question was.

4. Next, modify the code you've been building to include a metric `too_low` that is True when the guess was too low and False otherwise. Filter out the incorrect responses, and for each student see the proportion of times they guessed to low. Sort again by the proportion. Find yourself on the list.

5. Repeat what you did above, but do it by question. Do any of the responses suprise you?

6. We want to run an ANOVA test to see if there is a statistically significant relationship between whether the answer was correct and the student. To do this we need to add a with_column statement that converts the data type of the correct indicator from a Boolean value to a numeric one. You will need `correct = c.correct.cast(pl.Int64)`. Then, run the ANOVA and interpret the results.

7. Repeat the previous question, but run ANOVA with the `question_desc` variable

### Game 2: Remembering words

8. We need to do some cleaning of the next dataset as well to remove answers where someone responded twice with the same guess.

In [ ]:
g2 = (
    g2_orig
    .group_by(c.student, c.guess)
    .first()
    .sort(c.student, c.guess)
)
g2

9. Now, join `g2` to `g2_meta`. Note that these use different key names. Group by student, and compute a variable called `num_remembered` equal to the number of words the student remembered. Save the output as a dataset called `g2_student` and then print it out. We will need this later.

10. Join `g2` and `g2_meta` again. This time, group by `guess` and compute the number of guesses for each word divided by 15 (the number of students who responded). Sort by the score.

11. Repeat the previous task, but now group by guess and part of speech. Run a two-sample T-test to see if there is a statistically significant different between the scores of adjectives and the scores of nouns.

### Game 3: Video Memory

12. Create a dataset called `g3_student` that has one row for each student in the `g3` dataset and a column called `score` that gives the proportion of responses equal to "yes".

13. Join `g2_student` to `g3_student`. Plot the "num_remembered" on the x-axis and the "score" on the y-axis. Add a linear regression line.

14. Finally, compute the regression you have above using `DSStatsmodels.ols`. Do you see a statistically significant affect between the two variables?